# TeenPattiBench evaluation — Gemma 3 4B

Runs **Gemma 3 4B Instruct** over all 2,000 TeenPattiBench eval items and writes a
predictions file in the **standard Phase 4 format**, the same schema as
`results/preds_{qwen3_8b,qwen3_4b,mistral_7b,llama3.2_3b}_kaggle.jsonl`.

**Same setup as those four runs**, taken from their metadata headers:

| | |
|---|---|
| precision | `torch.float16`, unquantised |
| decoding | greedy (`do_sample=False`), `max_new_tokens=16`, no thinking |
| prompt | chat template, system prompt as a `system` turn, `add_generation_prompt=True` |
| batching | batch size 8, left padding |
| answers | first action word in the completion, **not** restricted to the legal set |
| output | metadata header + one `{"id", "answer"}` line per item |

Only the model changes.

### Before you run
1. **Runtime → Change runtime type → L4 GPU.** Do this *first* — switching the
   runtime wipes any files already uploaded.
2. Gemma is a **gated** model. Accept the licence at
   <https://huggingface.co/google/gemma-3-4b-it>, then add your token as a Colab secret
   named `HF_TOKEN` (🔑 in the left sidebar), or paste it when cell 4 asks.
3. **Upload three files** with the Files panel (📁 in the left sidebar) into `/content`:
   - `teenpattibench_eval.jsonl` — or `teenpattibench_eval_shuffled.jsonl` for the
     action-order control run (then set `EVAL_FILE` in cell 1 to match)
   - `teenpattibench_system_prompt.txt`
   - `metrics.py`
4. **Runtime → Run all.**

No data is downloaded from the internet. The only network use is the Gemma model
weights (from Hugging Face; too large to upload) and, only if Colab's preinstalled
`transformers` is older than 4.50, a package upgrade.

## 1. Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao"])

In [ ]:
MODEL_ID   = "google/gemma-3-4b-it"
MODEL_NAME = "Gemma 3 4B"
OUT        = "preds_gemma3_4b_colab.jsonl"

# --- identical to the Qwen3-8B / Qwen3-4B / Mistral-7B / Llama-3.2-3B runs ---
DTYPE          = "float16"   # kept for parity with the four earlier runs (T4, float16).
                            # L4 also supports "bfloat16" -- Gemma's native precision --
                            # but that is a deviation from those runs; use it only if the
                            # cell-7 smoke test fails, and it is recorded in the metadata.
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 16
DO_SAMPLE      = False
THINKING       = False      # Gemma has no thinking mode; recorded for schema parity

LIMIT = None                # e.g. 40 for a quick trial; None = all 2,000

# Files you uploaded to Colab's Files panel (left sidebar) before running.
# For the action-order control run, set EVAL_FILE = "teenpattibench_eval_shuffled.jsonl".
DATA_DIR    = "/content"
EVAL_FILE   = "teenpattibench_eval.jsonl"
SYSTEM_FILE = "teenpattibench_system_prompt.txt"

## 2. Check packages and the GPU

In [ ]:
import importlib.metadata as md
from packaging import version

# Colab ships transformers preinstalled. Gemma 3 needs >= 4.50; upgrade only if older.
try:
    tv = md.version("transformers")
except md.PackageNotFoundError:
    tv = "0"
if version.parse(tv) < version.parse("4.50.0"):
    print(f"transformers {tv} is too old for Gemma 3 -- upgrading")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                           "transformers>=4.50", "accelerate"])
else:
    print(f"transformers {tv} already installed -- no download needed")

import torch, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.")

TORCH_DTYPE = getattr(torch, DTYPE)
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS, "| dtype:", TORCH_DTYPE)

## 3. Load the files you uploaded

In [ ]:
import json, os, sys

os.chdir(DATA_DIR)
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

need = [EVAL_FILE, SYSTEM_FILE, "metrics.py"]
missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    present = sorted(f for f in os.listdir(DATA_DIR) if not f.startswith("."))
    raise SystemExit(
        f"Missing from {DATA_DIR}: {missing}\n"
        f"Found there: {present}\n"
        "Upload them with the Files panel (folder icon, left sidebar) and re-run this cell.\n"
        "If a file shows up as e.g. 'metrics (1).py', rename it or delete the duplicate."
    )

EVAL_PATH   = os.path.join(DATA_DIR, EVAL_FILE)
SYSTEM_PATH = os.path.join(DATA_DIR, SYSTEM_FILE)

# A shuffled-eval run gets its own output name, so it can never overwrite the v1 run.
SHUFFLED = "shuffled" in EVAL_FILE
if SHUFFLED and "shuffled" not in OUT:
    OUT = OUT.replace(".jsonl", "_shuffled.jsonl")

SYSTEM_PROMPT = open(SYSTEM_PATH, encoding="utf-8").read()
items = [json.loads(l) for l in open(EVAL_PATH, encoding="utf-8") if l.strip()]
if LIMIT:
    items = items[:LIMIT]
if not LIMIT and len(items) != 2000:
    raise SystemExit(f"Expected 2,000 eval items, found {len(items):,} -- wrong file?")

print(f"eval file : {EVAL_FILE}  ({'SHUFFLED action order' if SHUFFLED else 'v1, original order'})")
print(f"items     : {len(items):,}")
print(f"prompt    : {len(SYSTEM_PROMPT)} chars")
print(f"output    : {OUT}")

## 4. Hugging Face login (Gemma is gated)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in with the HF_TOKEN Colab secret.")
except Exception:
    login()

## 5. Load the model

The earlier runs loaded with `AutoModelForCausalLM`, so that is tried first.
`gemma-3-4b-it` is packaged as a vision-language checkpoint, and some `transformers`
versions will only load it through `Gemma3ForConditionalGeneration`; that is the
fallback. The class that loaded is recorded in the output metadata, just as
`LlamaForCausalLM` etc. are for the other runs.

In [ ]:
import importlib
from packaging import version
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# `dtype=` replaced `torch_dtype=` in transformers 4.56. Older versions silently
# ignore an unknown `dtype=` and load in float32, which does not fit on a T4.
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") \
           else "torch_dtype"

model = None
for cls_name in ("AutoModelForCausalLM", "Gemma3ForConditionalGeneration"):
    try:
        cls = getattr(importlib.import_module("transformers"), cls_name)
        model = cls.from_pretrained(MODEL_ID, device_map={"": 0}, **{DTYPE_KW: TORCH_DTYPE})
        print(f"loaded with {cls_name}")
        break
    except Exception as e:
        print(f"  {cls_name}: {type(e).__name__}: {str(e)[:120]}")
if model is None:
    raise SystemExit("Could not load the model.")
if model.dtype != TORCH_DTYPE:
    raise SystemExit(f"Model loaded as {model.dtype}, expected {TORCH_DTYPE}.")
model.eval()

# Greedy decoding. Clear the checkpoint's own sampling defaults so they are neither
# applied nor warned about -- the metadata records them as null, like the other runs.
gc = model.generation_config
gc.do_sample = DO_SAMPLE
for k in ("temperature", "top_p", "top_k"):
    if hasattr(gc, k):
        setattr(gc, k, None)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters on {model.device}")

## 6. Prompt construction and answer extraction

**Prompt.** System prompt as a `system` turn, the item's prompt as the `user` turn,
`add_generation_prompt=True` — as in the earlier runs. If a tokenizer rejects a
`system` role outright, the system text is folded into the user turn so the run can
proceed; that is announced loudly and recorded in the metadata, because it is a
different delivery from the other four runs.

**Answer.** The first TeenPatti action word in the completion, matched against the
*full* action vocabulary. The earlier runs did not filter by legality — Mistral-7B's
file contains `raise` 15 times where the raise cap made it illegal — so filtering
here would score this model more generously than those four. If no action word is
found, the stripped completion is kept and `metrics.py` scores it wrong.

In [ ]:
import re

def _messages(user_text, fold_system):
    if fold_system:
        return [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_text}]
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text}]

try:
    tok.apply_chat_template(_messages("probe", False), tokenize=False,
                            add_generation_prompt=True)
    FOLD_SYSTEM = False
    print("system role accepted -- same prompt delivery as the earlier runs")
except Exception as e:
    FOLD_SYSTEM = True
    print("!" * 78)
    print(f"This tokenizer rejects a system role ({type(e).__name__}).")
    print("The system prompt is folded into the user turn. This DIFFERS from the")
    print("earlier runs and is recorded in the output metadata.")
    print("!" * 78)

def build_prompt(user_text):
    return tok.apply_chat_template(_messages(user_text, FOLD_SYSTEM),
                                   tokenize=False, add_generation_prompt=True)

ACTION_WORDS = ("stay-blind", "see", "chaal", "raise", "pack", "show")

def extract_answer(completion):
    text = completion.strip().lower()
    hits = [(m.start(), w) for w in ACTION_WORDS
            for m in re.finditer(rf"(?<![a-z-]){re.escape(w)}(?![a-z-])", text)]
    return min(hits)[1] if hits else text

@torch.no_grad()
def generate(user_texts):
    # The chat template already emits the BOS token, so do not add another one.
    enc = tok([build_prompt(t) for t in user_texts], return_tensors="pt",
              padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                         pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n--- rendered prompt (first item, first 300 chars) ---")
print(build_prompt(items[0]["prompt"])[:300])

## 7. Smoke test

Five items before committing to 2,000. Gemma was trained in bfloat16 and can
overflow in float16; if it does, the completions come back empty or garbled and
this cell stops the run instead of wasting half an hour.

In [ ]:
probe = items[:5]
raws = generate([i["prompt"] for i in probe])
ok = 0
for it, raw in zip(probe, raws):
    ans = extract_answer(raw)
    ok += ans in ACTION_WORDS
    print(f"  {it['id']}  raw={raw.strip()[:40]!r:44}  answer={ans!r:14} legal={it['legal_actions']}")

first = tok(build_prompt(probe[0]["prompt"]), add_special_tokens=False)["input_ids"]
if tok.bos_token_id is not None and first[:2] == [tok.bos_token_id] * 2:
    raise SystemExit("Double BOS token in the input -- the prompt would be malformed.")

if ok == 0:
    raise SystemExit(
        "None of the 5 probes produced an action word. This is almost always Gemma\n"
        "overflowing in float16. On this L4, set DTYPE = 'bfloat16' in cell 1 and\n"
        "Run all again. That is recorded in the metadata as a deviation from the\n"
        "earlier runs' float16 -- mention it when reporting the result."
    )
print(f"\n{ok}/5 produced an action word -- proceeding.")

## 8. Run all items

In [ ]:
import time
from tqdm.auto import tqdm

records, t0 = [], time.time()
for s in tqdm(range(0, len(items), BATCH_SIZE), desc="generating"):
    batch = items[s:s + BATCH_SIZE]
    for it, raw in zip(batch, generate([i["prompt"] for i in batch])):
        records.append({"id": it["id"], "answer": extract_answer(raw)})

print(f"{len(records):,} predictions in {(time.time() - t0) / 60:.1f} min")

## 9. Write the predictions file

Same schema and key order as the four existing prediction files. The one possible
addition is `prompt.system_prompt_delivery`, written **only** if the system role had
to be folded into the user turn, so a deviation cannot pass silently.

In [ ]:
meta = {
    "type": "metadata",
    "model": {
        "name": MODEL_NAME,
        "model_id": MODEL_ID,
        "architecture": list(model.config.architectures or [type(model).__name__]),
        "model_type": model.config.model_type,
        "num_parameters": int(sum(p.numel() for p in model.parameters())),
        "dtype": str(model.dtype),
        "device": str(model.device),
        "quantized": False,
    },
    "tokenizer": {
        "name_or_path": tok.name_or_path,
        "vocab_size": tok.vocab_size,
        "pad_token": tok.pad_token,
        "eos_token": tok.eos_token,
        "bos_token": tok.bos_token,
    },
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "temperature": None,
        "top_p": None,
        "top_k": None,
        "thinking_enabled": THINKING,
    },
    "dataset": {
        "name": "TeenPattiBench",
        "evaluation_file": EVAL_PATH,
        "num_questions": len(items),
    },
    "prompt": {
        "system_prompt_file": SYSTEM_PATH,
        "system_prompt": SYSTEM_PROMPT,
    },
    "runtime": {
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_count": torch.cuda.device_count(),
        "gpus": GPUS,
    },
    "output": {
        "format": "JSONL",
        "prediction_file": os.path.abspath(OUT),
    },
}
if FOLD_SYSTEM:
    meta["prompt"]["system_prompt_delivery"] = "folded_into_user_turn"

with open(OUT, "w", encoding="utf-8") as fh:
    fh.write(json.dumps(meta) + "\n")
    for r in records:
        fh.write(json.dumps(r) + "\n")
print(f"wrote {OUT}: 1 metadata line + {len(records):,} predictions")

## 10. Check the file

In [ ]:
import collections

lines = [json.loads(l) for l in open(OUT, encoding="utf-8")]
header, recs = lines[0], lines[1:]
by_id = {i["id"]: i for i in items}
ids = [r["id"] for r in recs]

print("header type           :", header["type"])
print("records               :", len(recs))
print("unique ids            :", len(set(ids)) == len(recs))
print("covers every item     :", set(ids) == set(by_id))
print("record keys           :", sorted(set().union(*map(set, recs))))

dist = collections.Counter(r["answer"] for r in recs)
print("answers               :", dict(dist.most_common()))
off_vocab = {a: n for a, n in dist.items() if a not in ACTION_WORDS}
if off_vocab:
    print("  not an action word  :", off_vocab, "(kept verbatim; scored wrong)")
illegal = sum(1 for r in recs if r["answer"] in ACTION_WORDS
              and r["answer"] not in by_id[r["id"]]["legal_actions"])
print("action words illegal for their item:", illegal)

## 11. Score

The repository's own scorer, `src/metrics.py`. It reads `action`, so `answer` is
mapped across first — exactly how the four earlier files are scored.

In [ ]:
import metrics
preds = {r["id"]: {"action": r["answer"]} for r in recs}
report = metrics.score(items, preds)
print(metrics.format_report(report, metrics.trivial_baselines(items)))

## 12. Download

In [ ]:
from google.colab import files
files.download(OUT)
print(f"Commit as results/{OUT}, alongside the other four prediction files.")